# Azure ML + Dask

## Setup cluster

In [ ]:
from azureml.contrib.dask import AMLCluster

from azureml.core import Workspace

ws = Workspace.from_config()            # get workspace
ct = ws.compute_targets['dask-cluster'] # any Azure ML cluster 

cluster = AMLCluster(ct)  # only required input

## Cluster inputs

In [ ]:
cluster = AMLCluster(
      ct
    , use_gpu         = False # GPU compute target?
    , n_gpus_per_node = 0     # GPUs per node 
    , node_count      = None  # number of nodes if adapt set to False; default = 1
    , env             = None  # follow typical Azure ML environment specification options 
    , jupyter         = True  # start Jupyter Lab session on headnode
    , jupyter_token   = None  # default to random GUID
    , jupyter_port    = None  # default to 8888 
    , dashboard_port  = None  # default to 8788 
    , codestore       = None  # default mount the file share used by Compute Instance 
    , datastores      = None  # list of datastores to mount 
)

## Get links

In [ ]:
cluster.get_jupyter_link()

In [ ]:
cluster.get_dashboard_link()

## Scale cluster 

In [ ]:
cluster.scale(n=10)      # scale to 10 nodes, client adjust automatically 

In [ ]:
cluster.scale(n=1)       # scale to 1 node 

## Get run object

In [ ]:
run = cluster.get_run()  # regular Azure ML Run object with link to studio, logs, etc

## Get logs

Logs are available in the Azure ML studio in the associated run. This can basically be a wrapper instead of running cluster.get_run().get_logs().

In [ ]:
cluster.logs()           # match existing method 

## Update pip packages

In [ ]:
packages = 'dask[complete] matplotlib azureml-dataprep==0.1.3'

# update the cluster
cluster.install_pip_packages(packages) # magically install new packages on cluster 
cluster.restart()

## Read data

Users can either use built-in ADLS connectors for Gen 1 or 2, or use Azure ML Datasets.

In [ ]:
ds = ws.datasets['bigdata'] # get tabular dataset
df = ds.to_dask_dataframe() # get dask dataframe 
df.describe().compute()     # profile the data on the cluster 

## Process data

By default, dask.dataframe, dask.numpy, and other packages will use the cluster. Users can use delayed functions to execute any code on the cluster.

In [ ]:
df['temperature'] = df['temperature']*(9/5) + 32
# other data prep code

## Output dataset

In [ ]:
new_ds = Dataset.Tabular.from_dask_dataframe(df)